# Data Quality & Model Readiness

Before modelling, a quant needs to verify data integrity and understand potential pitfalls:

1. **Missing values** — where and how much?
2. **Train/test distribution shift** — is the test period structurally different?
3. **Feature distributions** — outliers, scale differences, structural breaks
4. **Plant metadata & unavailability** — capacity constraints and outage patterns

Back to [main EDA](eda.ipynb)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd

from src.data import load_train, load_test, load_plant_metadata, load_unavailability
from src.plotting import *

train = load_train()
test = load_test()
metadata = load_plant_metadata()
prod_unavail = load_unavailability('prod')
cons_unavail = load_unavailability('cons')

## 1. Missing Values

Any missing data needs to be handled before modelling. Check both train and test — if test has missing features that train doesn't, forward-fill or interpolation strategies need to be planned.

In [ ]:
train_missing = train.isnull().sum()
test_missing = test.isnull().sum()

train_any = train_missing[train_missing > 0]
test_any = test_missing[test_missing > 0]

if not train_any.empty:
    print("=== Train Missing ===")
    for col, n in train_any.items():
        print(f"  {col}: {n} ({n/len(train):.2%})")
else:
    print("Train: no missing values.")

print()

if not test_any.empty:
    print("=== Test Missing ===")
    for col, n in test_any.items():
        print(f"  {col}: {n} ({n/len(test):.2%})")
else:
    print("Test: no missing values.")

In [ ]:
fig = plot_missing_values(train, title="Missing Values — Train")
if fig:
    fig.show()

fig = plot_missing_values(test, title="Missing Values — Test")
if fig:
    fig.show()

## 2. Train / Test Distribution Shift

The test period (Aug 2024 → Feb 2025) follows the train period (Dec 2022 → Jul 2024). Key risks:
- **Seasonal shift:** test covers summer → winter, while train covers the full cycle. Winter-specific patterns may be underrepresented.
- **Structural changes:** gas price cap expiry (Dec 2023), market reforms, new capacity.

We check whether key feature distributions have shifted between train and test.

In [ ]:
shift_cols = [
    'es_demand_f_d1', 'es_wind_f_d1', 'es_solar_f_d1',
    'es_temp_f_d1', 'es_gas_market_price_d1',
]
for col in shift_cols:
    plot_train_test_split(train, test, columns=[col]).show()

In [ ]:
# Quantitative comparison: train vs test summary stats
common_cols = [c for c in shift_cols if c in train.columns and c in test.columns]
comparison = pd.DataFrame({
    'train_mean': train[common_cols].mean(),
    'test_mean': test[common_cols].mean(),
    'train_std': train[common_cols].std(),
    'test_std': test[common_cols].std(),
})
comparison['mean_shift_%'] = ((comparison['test_mean'] - comparison['train_mean']) / comparison['train_mean'] * 100).round(1)
comparison['std_shift_%'] = ((comparison['test_std'] - comparison['train_std']) / comparison['train_std'] * 100).round(1)
comparison.round(1)

## 3. Temporal Completeness

Check for gaps in the hourly time series. Missing hours would break lag features and rolling calculations.

In [ ]:
for name, df in [('Train', train), ('Test', test)]:
    dt = df['datetime_start']
    diffs = dt.diff().dt.total_seconds() / 3600
    gaps = diffs[diffs != 1.0].dropna()
    if gaps.empty:
        print(f"{name}: continuous hourly series, no gaps.")
    else:
        print(f"{name}: {len(gaps)} gaps detected:")
        for idx, gap_h in gaps.items():
            print(f"  {dt.iloc[idx-1]} → {dt.iloc[idx]}  ({gap_h:.0f}h gap)")

## 4. Feature Distributions by Category

Checking for outliers, scale differences, and unexpected patterns within each feature group.

In [ ]:
for group_name, cols in FEATURE_GROUPS.items():
    existing = [c for c in cols if c in train.columns]
    if existing:
        plot_boxplots(train, existing, title=f"{group_name.title()} — Distributions").show()

## 5. Structural Breaks — Gas Price Cap

The Iberian gas price cap was active until Dec 2023. This artificially suppressed gas prices in Spain. After expiry, gas prices returned to market levels. This is a known structural break that may affect model generalisation.

In [ ]:
import plotly.graph_objects as go

fig = plot_timeseries(train, ['es_gas_market_price_d1'], rolling_window=168,
                      title='Gas Price — Structural Break at Cap Expiry')

# Mark the gas price cap expiry
fig.add_shape(type='line', x0='2023-12-31', x1='2023-12-31', y0=0, y1=1,
              yref='paper', line=dict(dash='dash', color='red'))
fig.add_annotation(x='2023-12-31', y=1, yref='paper', text='Gas cap expiry',
                   showarrow=False, yanchor='bottom')
fig.show()

## 6. Plant Metadata & Capacity Constraints

The target is physically bounded by the fleet's total capacity. Understanding individual plant sizes helps interpret unavailability data.

In [ ]:
total_gen = (metadata['capacity_per_unit'] * metadata['units']).sum()
total_pump = (metadata['pump_load_per_unit'] * metadata['pump_units']).sum()

print(f"Fleet: {len(metadata)} plants, {metadata['units'].sum()} generation units, {metadata['pump_units'].sum()} pump units")
print(f"Total generation capacity: {total_gen:,.0f} MW")
print(f"Total pump load capacity:  {total_pump:,.0f} MW")
print(f"Pump efficiency:           {metadata['pump_efficiency'].iloc[0]:.0%}")
print()

# Top 5 plants by capacity
metadata_sorted = metadata.copy()
metadata_sorted['total_gen_cap'] = metadata_sorted['capacity_per_unit'] * metadata_sorted['units']
metadata_sorted['total_pump_cap'] = metadata_sorted['pump_load_per_unit'] * metadata_sorted['pump_units']
print("Top 5 plants by generation capacity:")
print(metadata_sorted.nlargest(5, 'total_gen_cap')[['unit_name', 'total_gen_cap', 'total_pump_cap']].to_string(index=False))

plot_plant_capacity(metadata).show()

## 7. Unavailability Data

When plants are offline for maintenance or breakdown, the available capacity drops. If large plants go offline, the target's range is physically constrained. This is a potential feature for the model.

In [ ]:
print(f"Production unavailability records: {len(prod_unavail)}")
print(f"Consumption unavailability records: {len(cons_unavail)}")
print()
print("Production unavailability by unit:")
print(prod_unavail.groupby('unit')['unavailable'].agg(['count', 'mean', 'max']).round(0).to_string())
print()
print("Consumption unavailability by unit:")
print(cons_unavail.groupby('unit')['unavailable'].agg(['count', 'mean', 'max']).round(0).to_string())

In [ ]:
plot_unavailability_timeline(prod_unavail, cons_unavail).show()

## 8. Correlation Between D1 and D2 Features

If D1 and D2 are near-identical, including both adds multicollinearity without information gain. If they diverge, the difference (forecast revision) might itself be a useful feature.

In [ ]:
d1_cols = [c for c in train.columns if c.endswith('_d1')]
d2_cols = [c.replace('_d1', '_d2') for c in d1_cols]

print(f"{'Feature base':30s} | corr(D1,D2) | D1 std / D2 std")
print("-" * 65)
for d1, d2 in zip(d1_cols, d2_cols):
    if d2 in train.columns:
        r = train[d1].corr(train[d2])
        ratio = train[d1].std() / train[d2].std() if train[d2].std() > 0 else float('nan')
        base = d1.replace('_d1', '')
        print(f"{base:30s} | {r:.4f}      | {ratio:.3f}")

---

## Data Quality Summary

| Check | Status | Notes |
|-------|--------|-------|
| Missing values | See above | Some features have ~0.2% missing — forward-fill is safe for hourly data |
| Temporal gaps | See above | Check for DST transitions |
| Train/test shift | Moderate | Test covers Aug–Feb (summer→winter), train covers full year |
| Structural breaks | Gas cap expiry Dec 2023 | Consider using post-cap data only, or adding a regime indicator |
| D1/D2 redundancy | High (>0.95 corr) | Prefer D1 only; consider D1−D2 diff as feature |
| Unavailability | Sparse, plant-specific | Could engineer total available capacity as a feature |

## 9. NTC Missingness Mechanism

NTC (Net Transfer Capacity) is the most heavily missing feature (~14% in train). Is the missingness random, or does it correlate with market stress (high demand, high prices)? If non-random, the missingness pattern itself is informative.

In [ ]:
from scipy.stats import chi2_contingency
import plotly.graph_objects as go

# NTC missingness indicator
train['ntc_missing'] = train['fr_es_ntc_d1'].isna().astype(int)
train['hour'] = train['datetime_start'].dt.hour
train['month'] = train['datetime_start'].dt.month

# 1. Missingness over time
monthly_miss = train.groupby(train['datetime_start'].dt.to_period('M'))['ntc_missing'].mean() * 100

fig = go.Figure(go.Bar(
    x=[str(p) for p in monthly_miss.index],
    y=monthly_miss.values,
    marker_color=COLORS['primary'],
))
apply_layout(fig, title='NTC Missingness Rate Over Time',
             xaxis_title='Month', yaxis_title='% Missing', height=400)
fig.update_xaxes(tickangle=-45)
fig.show()

# 2. Does missingness correlate with market variables?
market_vars = ['es_demand_f_d1', 'es_wind_f_d1', 'es_solar_f_d1', 'es_temp_f_d1']
print('Mean of market variables when NTC is missing vs present:')
for col in market_vars:
    present = train[train['ntc_missing'] == 0][col].mean()
    missing = train[train['ntc_missing'] == 1][col].mean()
    print(f'  {col:30s}: present={present:.1f}, missing={missing:.1f}, diff={missing-present:+.1f}')

# 3. Chi-squared: missingness vs high-demand indicator
train['high_demand'] = (train['es_demand_f_d1'] > train['es_demand_f_d1'].median()).astype(int)
ct = pd.crosstab(train['ntc_missing'], train['high_demand'])
chi2, p, dof, expected = chi2_contingency(ct)
print(f'\nChi-squared test (NTC missing vs high demand): chi2={chi2:.2f}, p={p:.4e}')
print('Interpretation: p < 0.05 → missingness is NOT random w.r.t. demand level')

## 10. Stationarity Tests

Non-stationary features can cause spurious correlations. We test key series with ADF (null: unit root) and KPSS (null: stationary). A series that fails both tests needs differencing or detrending.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

test_cols = ['es_demand_f_d1', 'es_wind_f_d1', 'es_solar_f_d1',
             'es_gas_market_price_d1', 'eua_price', 'es_total_ps']

results = []
for col in test_cols:
    series = train[col].dropna()
    if len(series) < 100:
        continue

    # ADF test (H0: unit root / non-stationary)
    adf_stat, adf_p, *_ = adfuller(series, maxlag=48, autolag='AIC')

    # KPSS test (H0: stationary)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        kpss_stat, kpss_p, *_ = kpss(series, regression='c', nlags='auto')

    # Interpretation
    if adf_p < 0.05 and kpss_p >= 0.05:
        verdict = 'Stationary'
    elif adf_p >= 0.05 and kpss_p < 0.05:
        verdict = 'Non-stationary'
    elif adf_p < 0.05 and kpss_p < 0.05:
        verdict = 'Trend-stationary'
    else:
        verdict = 'Inconclusive'

    results.append({
        'Feature': col,
        'ADF stat': adf_stat,
        'ADF p': adf_p,
        'KPSS stat': kpss_stat,
        'KPSS p': kpss_p,
        'Verdict': verdict,
    })

stationarity = pd.DataFrame(results).round(4)
print('Stationarity Tests (ADF H0: non-stationary, KPSS H0: stationary):')
stationarity

## 11. Structural Break Detection — Gas Price → Target Relationship

The gas price cap expired Dec 2023. We test whether the relationship between gas price and the target changed using rolling regression coefficients. A stable coefficient means the structural break is benign; an unstable one means the model may need separate regimes.

In [ ]:
from sklearn.linear_model import LinearRegression
from scipy.stats import f as f_dist

# Rolling regression: gas_price → target
train_clean = train[['datetime_start', 'es_gas_market_price_d1', 'es_total_ps']].dropna().copy()
train_clean = train_clean.sort_values('datetime_start').reset_index(drop=True)

window = 30 * 24  # 30 days
coefficients = []
dates = []

for i in range(window, len(train_clean)):
    sub = train_clean.iloc[i - window:i]
    X = sub[['es_gas_market_price_d1']].values
    y = sub['es_total_ps'].values
    lr = LinearRegression().fit(X, y)
    coefficients.append(lr.coef_[0])
    dates.append(train_clean.iloc[i]['datetime_start'])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dates, y=coefficients,
    mode='lines', name='Rolling β (gas→target)',
    line=dict(color=COLORS['primary'], width=1.5),
))
fig.add_shape(type='line', x0='2023-12-31', x1='2023-12-31', y0=min(coefficients), y1=max(coefficients),
              line=dict(dash='dash', color='red', width=2))
fig.add_annotation(x='2023-12-31', y=max(coefficients), text='Gas cap expiry',
                   showarrow=False, yanchor='bottom', font=dict(color='red'))
fig.add_hline(y=0, line_dash='dot', line_color='grey')
apply_layout(fig, title='Rolling 30-Day Regression Coefficient: Gas Price → Target',
             xaxis_title='Date', yaxis_title='β coefficient', height=450)
fig.show()

# Chow test equivalent: compare pre/post coefficients
breakpoint_idx = train_clean['datetime_start'].searchsorted(pd.Timestamp('2023-12-31', tz='UTC'))
pre_data = train_clean.iloc[:breakpoint_idx]
post_data = train_clean.iloc[breakpoint_idx:]

from sklearn.metrics import mean_squared_error

# Full model
X_full = train_clean[['es_gas_market_price_d1']].values
y_full = train_clean['es_total_ps'].values
lr_full = LinearRegression().fit(X_full, y_full)
rss_full = np.sum((y_full - lr_full.predict(X_full)) ** 2)

# Pre model
X_pre = pre_data[['es_gas_market_price_d1']].values
y_pre = pre_data['es_total_ps'].values
lr_pre = LinearRegression().fit(X_pre, y_pre)
rss_pre = np.sum((y_pre - lr_pre.predict(X_pre)) ** 2)

# Post model
X_post = post_data[['es_gas_market_price_d1']].values
y_post = post_data['es_total_ps'].values
lr_post = LinearRegression().fit(X_post, y_post)
rss_post = np.sum((y_post - lr_post.predict(X_post)) ** 2)

# Chow F-statistic
k = 2  # number of parameters (slope + intercept)
n = len(train_clean)
f_stat = ((rss_full - rss_pre - rss_post) / k) / ((rss_pre + rss_post) / (n - 2 * k))
f_p = 1 - f_dist.cdf(f_stat, k, n - 2 * k)

print(f'Chow test for structural break at Dec 2023:')
print(f'  F-statistic: {f_stat:.2f}')
print(f'  p-value: {f_p:.4e}')
print(f'  Pre-cap β: {lr_pre.coef_[0]:.3f}')
print(f'  Post-cap β: {lr_post.coef_[0]:.3f}')
print(f'  {"Significant" if f_p < 0.05 else "Not significant"} structural break at 5% level')